In [2]:
# Import required libraries  
import os  
import json  
import openai  
import pandas as pd    
from dotenv import load_dotenv 
from openai import AzureOpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt  
from azure.core.credentials import AzureKeyCredential  
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient  
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
) 

In [3]:
# Configure environment variables  
load_dotenv()  
# Get the Azure Credential
credential = DefaultAzureCredential()

# Set the API type to `azure_ad`
os.environ["OPENAI_API_TYPE"] = "azure_ad"
# Set the API_KEY to the token from the Azure credential
os.environ["OPENAI_API_KEY"] = credential.get_token("https://cognitiveservices.azure.com/.default").token
# Set the ENDPOINT
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://ztn-oai-fc.openai.azure.com/"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "https://ztn-oai-fc.openai.azure.com/"

# The following variables from your .env file are used in this notebook
endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_ADMIN_KEY", "")) if len(os.getenv("AZURE_SEARCH_ADMIN_KEY", "")) > 0 else DefaultAzureCredential()
index_name = os.getenv("AZURE_SEARCH_INDEX", "test-rag-traffic-analysis")
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.getenv("AZURE_OPENAI_KEY", "") if len(os.getenv("AZURE_OPENAI_KEY", "")) > 0 else None
azure_openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_embedding_dimensions = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", 1024))
embedding_model_name = "text-embedding-3-large"
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-06-01")

In [4]:
openai_credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(openai_credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
    azure_deployment=azure_openai_embedding_deployment,
    api_version=azure_openai_api_version,
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_key,
    azure_ad_token_provider=token_provider if not azure_openai_key else None
)
print(client)

## Generate Document Embeddings using text-embedding model (bug rasied with new azure search)

In [5]:

# Read the human created rag json
with open('data/rag_constraints.json', 'r', encoding='utf-8') as file:
    input_data = json.load(file)

labels = [item['label'] for item in input_data]
constraints = [item['constraint'] for item in input_data]

# TODO: why bug here??
# NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}
labels_response = client.embeddings.create(input=labels, model=azure_openai_embedding_deployment)

# TODO: if the above line of code works, then the following should work
# labels_embeddings = [item.embedding for item in labels_response.data]
# constraints_response = client.embeddings.create(input=constraints, model=embedding_model_name)
# constraints_embeddings = [item.embedding for item in constraints_response.data]

# # Generate embeddings for title and content fields
# for i, item in enumerate(input_data):
#     labels = item['label']
#     constraints = item['constraint']
#     item['labelVector'] = labels_embeddings[i]
#     item['constraintVector'] = constraints_embeddings[i]

# # Output the embeddings to docVectors.json file
# with open("output/constraintVectors.json", "w") as f:
#     json.dump(input_data, f)

NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

## Create a search index

In [6]:
# Create a search index
index_client = SearchIndexClient(
    endpoint=endpoint, credential=credential)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, sortable=True, filterable=True, facetable=True),
    SearchableField(name="label", type=SearchFieldDataType.String),
    SearchableField(name="constraint", type=SearchFieldDataType.String,
                    filterable=True),
    SearchField(name="labelVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=azure_openai_embedding_dimensions, vector_search_profile_name="myHnswProfile"),
    SearchField(name="constraintVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=azure_openai_embedding_dimensions, vector_search_profile_name="myHnswProfile"),
]

# Configure the vector search configuration  
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
            vectorizer="myVectorizer"
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            name="myVectorizer",
            azure_open_ai_parameters=AzureOpenAIParameters(
                resource_uri=azure_openai_endpoint,
                deployment_id=azure_openai_embedding_deployment,
                model_name=embedding_model_name,
                api_key=azure_openai_key
            )
        )
    ]
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        keywords_fields=[SemanticField(field_name="label")],
        content_fields=[SemanticField(field_name="constraint")]
    )
)

# Create the semantic settings with the configuration
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=index_name, fields=fields,
                    vector_search=vector_search, semantic_settings=semantic_search)
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 test-rag-traffic-analysis created


## Upload the constraintVectors.json file to the created search index

In [21]:
# Upload some documents to the index
with open('output/constraintVectors.json', 'r') as file:  
    documents = json.load(file)  
search_client = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents)  
print(f"Uploaded {len(documents)} documents")

Uploaded 6 documents
